In [ ]:
import numpy as np
import inspect


class BaseEstimator:
    def get_params(self, deep: bool = True):
        sig = inspect.signature(self.__init__)
        return {name: getattr(self, name)
                for name in sig.parameters if name != "self"}
    def set_params(self, **params):
        for k, v in params.items():
            if not hasattr(self, k):
                raise ValueError(f"Invalid parameter {k!r}")
            setattr(self, k, v)
        return self

def check_is_fitted(estimator, attributes=None):
    if attributes is None:
        attributes = [v for v in vars(estimator) if v.endswith("_")]
    if not all(hasattr(estimator, a) for a in attributes):
        raise AttributeError("This estimator is not fitted yet. Call 'fit' first.")

def _as_2d(X):
    X = np.asarray(X)
    if X.ndim == 1: X = X.reshape(-1, 1)
    if X.ndim != 2: raise ValueError("X must be 2D.")
    return X

def check_array(X, dtype=np.float64):
    X = _as_2d(X).astype(dtype, copy=False)
    if not np.all(np.isfinite(X)): raise ValueError("Input contains NaN or inf.")
    return X

def check_X_y(X, y, dtype=np.float64):
    X = check_array(X, dtype=dtype)
    y = np.asarray(y)
    if y.ndim != 1: y = y.reshape(-1)
    if X.shape[0] != y.shape[0]: raise ValueError("X and y size mismatch.")
    if not np.all(np.isfinite(y)): raise ValueError("y contains NaN or inf.")
    return X, y

def sigmoid(z):
    z = np.asarray(z, dtype=np.float64)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out


class LogisticRegressionScratch(BaseEstimator):
    """
    From-scratch, vectorized binary Logistic Regression with L2 penalty.
    Parameters
    ----------
    C : float
        Inverse of L2 regularization strength. Larger C -> weaker regularization.
        Here we scale the penalty as alpha = 1 / (C * n_samples) to match a mean loss.
    tol : float
        Convergence threshold on gradient norm.
    max_iter : int
        Maximum iterations.
    fit_intercept : bool
        Whether to fit an intercept.
    lr : float or None
        If None, uses backtracking line search; else constant step size.
    random_state : int or None
        Seed for weight init.
    verbose : bool
        Print progress every ~50 iters.
    """
    def __init__(self, C=1.0, tol=1e-6, max_iter=1000,
                 fit_intercept=True, lr=None, random_state=None, verbose=False):
        self.C = C
        self.tol = tol
        self.max_iter = max_iter
        self.fit_intercept = fit_intercept
        self.lr = lr
        self.random_state = random_state
        self.verbose = verbose

    def _loss_and_grad(self, X, y, w, b, alpha, update_intercept=True):
        n = X.shape[0]
        z = X @ w + (b if update_intercept else 0.0)
        # mean logistic loss: log(1+exp(z)) - y*z, evaluated stably with logaddexp
        loss_data = np.mean(np.logaddexp(0.0, z) - y * z)
        loss_reg = 0.5 * alpha * (w @ w)
        loss = loss_data + loss_reg
        p = sigmoid(z)
        grad_w = (X.T @ (p - y)) / n + alpha * w
        grad_b = np.mean(p - y) if update_intercept else 0.0
        return loss, grad_w, grad_b

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        classes = np.unique(y)
        if classes.size != 2:
            raise ValueError("Binary classification only in this implementation.")
        self.classes_ = classes
        y01 = (y == classes[1]).astype(np.float64)

        n_samples, n_features = X.shape
        self.n_features_in_ = n_features
        rng = np.random.default_rng(self.random_state)
        w = rng.normal(scale=0.01, size=n_features)
        b = 0.0
        alpha = 0.0 if (self.C is None or self.C == np.inf) else 1.0 / (self.C * n_samples)

        def loss_fn(w, b):
            z = X @ w + (b if self.fit_intercept else 0.0)
            return np.mean(np.logaddexp(0.0, z) - y01 * z) + 0.5 * alpha * (w @ w)

        lr = self.lr
        for i in range(int(self.max_iter)):
            loss, grad_w, grad_b = self._loss_and_grad(
                X, y01, w, b, alpha, update_intercept=self.fit_intercept
            )
            grad_norm = np.sqrt(grad_w @ grad_w + (grad_b * grad_b if self.fit_intercept else 0.0))
            if grad_norm < self.tol:
                if self.verbose:
                    print(f"Converged at iter {i}: grad_norm={grad_norm:.3e}, loss={loss:.6f}")
                break

            step = lr if lr is not None else 1.0
            if lr is None:
                # Armijo backtracking line search
                c, beta, max_bk = 1e-4, 0.5, 20
                for _ in range(max_bk):
                    w_new = w - step * grad_w
                    b_new = b - step * grad_b if self.fit_intercept else b
                    new_loss = loss_fn(w_new, b_new)
                    if new_loss <= loss - c * step * grad_norm * grad_norm:
                        break
                    step *= beta
                else:
                    step = 1e-6

            w = w - step * grad_w
            if self.fit_intercept:
                b = b - step * grad_b

            if self.verbose and (i % 50 == 0 or i == self.max_iter - 1):
                print(f"iter {i:4d}  loss={loss:.6f}  step={step:.2e}  grad_norm={grad_norm:.3e}")

        self.coef_ = w.reshape(1, -1)
        self.intercept_ = np.array([b]) if self.fit_intercept else np.array([0.0])
        self.n_iter_ = i + 1
        return self

    def decision_function(self, X):
        check_is_fitted(self, ["coef_", "intercept_"])
        X = check_array(X)
        if X.shape[1] != self.coef_.shape[1]:
            raise ValueError(f"X has {X.shape[1]} features, expected {self.coef_.shape[1]}.")
        return X @ self.coef_[0] + (self.intercept_[0] if self.fit_intercept else 0.0)

    def predict_proba(self, X):
        z = self.decision_function(X)
        p1 = sigmoid(z)
        return np.column_stack([1.0 - p1, p1])

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    def score(self, X, y):
        return np.mean(self.predict(X) == y)



In [ ]:
if __name__ == "__main__":
    rng = np.random.default_rng(0)
    n = 2000
    X = rng.normal(size=(n, 2))
    y = (X[:, 0] + 0.7 * X[:, 1] + rng.normal(scale=0.5, size=n) > 0).astype(int)
    perm = rng.permutation(n)
    Xtr, Xte = X[perm[:1600]], X[perm[1600:]]
    ytr, yte = y[perm[:1600]], y[perm[1600:]]
    clf = LogisticRegressionScratch(C=1.0, lr=None, max_iter=300, tol=1e-6, random_state=42)
    clf.fit(Xtr, ytr)
    print("acc:", clf.score(Xte, yte), "iters:", clf.n_iter_)


acc: 0.8975 iters: 300
